In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-8

In [ ]:
h = 5

w_range = np.arange(0.5, 3.5, 0.1)

In [ ]:
w = 2

In [ ]:
def analytic_scale(w, h):
    return (2 * (h - w) / np.pi + w) / h

In [ ]:
h = 5
w = 3
res = 500

triArea = h * w / res
avg_len = triArea ** 0.5

In [ ]:
import igl

In [ ]:
ipu, points, segment_edges, m, marker= periodic_unit_helper.get_zigzag_tube(h, w, avg_len)

In [ ]:
visualization.plot_line_segments(points, segment_edges)

In [ ]:
visualization.plot_2d_mesh(m, pointList=np.where(marker)[0], width=5, height=5)

In [ ]:
import periodic_unit_helper

In [ ]:
fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.showWireframe(True)

In [ ]:
ipu.sheet.rigidMotionPinVars

In [ ]:
fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

In [ ]:
# ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 0.5
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
benchmark.report()

In [ ]:
import compute_vibrational_modes

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ipu, fixedVars=[], mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-6)


In [ ]:
# ipu.sheet.setUseTensionFieldEnergy(False)
# ipu.sheet.setUseHessianProjectedEnergy(True)

In [ ]:
import mode_viewer, importlib
importlib.reload(mode_viewer);
mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=20)
mview.show()

### Experiment 2

In [ ]:
analytic_sfs = []
simulated_sfs = []
counter = 0
for w in w_range:
    print("w: ", w)
    print("analytic scale factor: ", analytic_scale(w, h))
    analytic_sfs.append(analytic_scale(w, h))
    
    import igl

    ipu, _, _, _, _ = periodic_unit_helper.get_zigzag_tube(h, w, 0.05)

    fixedVars = periodic_unit_helper.get_center_fixedVars(ipu)

    # isheet.setRelaxedStiffnessEpsilon(1e-6)

    from tri_mesh_viewer import TriMeshViewer
    viewer = TriMeshViewer(ipu, width=768, height=640)
    viewer.showWireframe(True)
    ipu.sheet.rigidMotionPinVars

    fd_perturb = np.random.uniform(-1e-3, 1e-3, ipu.numVars())

    # ipu.setVars(ipu.getVars() + fd_perturb)

    import time, vis
    ipu.sheet.setUseTensionFieldEnergy(True)
    ipu.sheet.setUseHessianProjectedEnergy(False)
    ipu.sheet.pressure = 0.5
    opts.niter = 100
    framerate = 5 # Update every 5 iterations
    def cb(it):
        if it % framerate == 0:
            viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb)
    
    render = viewer.offscreenRenderer(1000, 1000)
    render.render()
    render.save("zigzag_inflated_with_width_{}.png".format(counter))
    counter += 1
    sfs = periodic_unit_helper.get_deformation_scale_factors(ipu)
    print("Simulated scale factor: ", sfs)
    simulated_sfs.append(sfs)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, ax = plt.subplots()
plt.plot(w_range, analytic_sfs, '-o', label="Analytic scale factors")
plt.plot(w_range, simulated_sfs, '-o', label="Simulated scale factors")
ax.legend()
plt.xlabel("wall width")
plt.ylabel("scale factor")
plt.savefig('zigzag_scale_factor_comparison.png', dpi=300)